# P3.09 Discovery: Runtime Packages Probe

**PURPOSE:** Validate availability of critical runtime dependencies (FFmpeg, ffprobe, Node.js, npm, Python packages)

**TIMEOUT:** ≤5 minutes

**CRITICAL:** P3.09 render worker needs FFmpeg for H.264 encoding; other tools are optional.

In [ ]:
import json
import time
import subprocess
import sys
from datetime import datetime
from pathlib import Path

DISCOVERY_SESSION = {
    "session_id": f"runtime_probe_{int(time.time())}",
    "timestamp": datetime.now().isoformat(),
    "timeout_hard": 300,
    "timeout_warning": 270,
    "notebook_name": "kaggle_discovery_03_runtime_packages_probe",
    "results": []
}

start_time = time.time()

def log_test(test_name, result, evidence, duration_s):
    DISCOVERY_SESSION["results"].append({
        "test": test_name,
        "result": result,
        "evidence": evidence,
        "duration_s": duration_s,
        "timestamp": datetime.now().isoformat()
    })
    print(f"[{result:8s}] {test_name} ({duration_s:.1f}s)")

def check_timeout():
    elapsed = time.time() - start_time
    if elapsed > DISCOVERY_SESSION["timeout_hard"]:
        raise RuntimeError(f"HARD TIMEOUT: {elapsed:.0f}s")
    return elapsed

def probe_command(cmd_list, timeout_s=5):
    """Run a command and return success/failure with version info."""
    try:
        result = subprocess.run(cmd_list, capture_output=True, text=True, timeout=timeout_s)
        if result.returncode == 0:
            # Extract version from output (usually first line)
            output_lines = result.stdout.split("\n")
            version_line = output_lines[0] if output_lines else ""
            return {"available": True, "version": version_line[:100], "returncode": 0}
        else:
            return {"available": False, "error": result.stderr[:100], "returncode": result.returncode}
    except FileNotFoundError:
        return {"available": False, "error": "Command not found in PATH"}
    except subprocess.TimeoutExpired:
        return {"available": False, "error": "Command timeout"}
    except Exception as e:
        return {"available": False, "error": str(e)[:100]}

print(f"🔬 P3.09 Runtime Packages Probe: {DISCOVERY_SESSION['session_id']}")
print(f"📍 Started: {DISCOVERY_SESSION['timestamp']}")

## Test 1: FFmpeg Availability

In [ ]:
test_start = time.time()
check_timeout()

try:
    ffmpeg_probe = probe_command(["ffmpeg", "-version"])
    
    # FFmpeg is CRITICAL for P3.09
    result_status = "PASS" if ffmpeg_probe["available"] else "FAIL"
    duration = time.time() - test_start
    log_test("ffmpeg_availability", result_status, ffmpeg_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("ffmpeg_availability", "FAIL", str(e)[:100], duration)

## Test 2: ffprobe Availability

In [ ]:
test_start = time.time()
check_timeout()

try:
    ffprobe_probe = probe_command(["ffprobe", "-version"])
    
    # ffprobe is useful for validation
    result_status = "PASS" if ffprobe_probe["available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("ffprobe_availability", result_status, ffprobe_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("ffprobe_availability", "FAIL", str(e)[:100], duration)

## Test 3: Node.js Availability

In [ ]:
test_start = time.time()
check_timeout()

try:
    node_probe = probe_command(["node", "--version"])
    
    # Node.js optional for P3.09 (Remotion is via Python/Node bridge)
    result_status = "PASS" if node_probe["available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("nodejs_availability", result_status, node_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("nodejs_availability", "FAIL", str(e)[:100], duration)

## Test 4: npm Availability

In [ ]:
test_start = time.time()
check_timeout()

try:
    npm_probe = probe_command(["npm", "--version"])
    
    result_status = "PASS" if npm_probe["available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("npm_availability", result_status, npm_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("npm_availability", "FAIL", str(e)[:100], duration)

## Test 5: Critical Python Packages

In [ ]:
test_start = time.time()
check_timeout()

try:
    python_packages_probe = {
        "packages_checked": [],
        "packages_available": []
    }
    
    # Check critical packages
    packages_to_check = ["numpy", "Pillow", "requests", "subprocess", "json"]
    
    for pkg in packages_to_check:
        python_packages_probe["packages_checked"].append(pkg)
        try:
            __import__(pkg)
            python_packages_probe["packages_available"].append(pkg)
        except ImportError:
            pass
    
    python_packages_probe["availability_percentage"] = (
        len(python_packages_probe["packages_available"]) / len(python_packages_probe["packages_checked"]) * 100
    )
    
    result_status = "PASS" if python_packages_probe["availability_percentage"] >= 80 else "UNKNOWN"
    duration = time.time() - test_start
    log_test("python_packages", result_status, python_packages_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("python_packages", "FAIL", str(e)[:100], duration)

## Test 6: Remotion via Node.js

In [ ]:
test_start = time.time()
check_timeout()

try:
    remotion_probe = {
        "node_available": False,
        "npm_available": False,
        "remotion_path_check": "NOT_CHECKED",
        "verdict": "UNKNOWN"
    }
    
    # Check if Node.js and npm are available (prerequisites for Remotion)
    try:
        result = subprocess.run(["node", "--version"], capture_output=True, timeout=3)
        remotion_probe["node_available"] = result.returncode == 0
    except:
        pass
    
    try:
        result = subprocess.run(["npm", "--version"], capture_output=True, timeout=3)
        remotion_probe["npm_available"] = result.returncode == 0
    except:
        pass
    
    # Check if Remotion CLI is available (if Node.js/npm present)
    if remotion_probe["node_available"] and remotion_probe["npm_available"]:
        try:
            result = subprocess.run(["npm", "list", "-g", "remotion"], capture_output=True, text=True, timeout=5)
            remotion_probe["remotion_path_check"] = "AVAILABLE" if result.returncode == 0 else "NOT_INSTALLED"
            remotion_probe["verdict"] = "Remotion installable via npm" if remotion_probe["npm_available"] else "BLOCKED"
        except:
            remotion_probe["remotion_path_check"] = "ERROR_CHECKING"
    
    result_status = "PASS" if remotion_probe["node_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("remotion_prerequisites", result_status, remotion_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("remotion_prerequisites", "FAIL", str(e)[:100], duration)

## Final Report

In [ ]:
results = DISCOVERY_SESSION["results"]
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
unknown = sum(1 for r in results if r["result"] == "UNKNOWN")

total_time = time.time() - start_time

# Critical: FFmpeg must be present
ffmpeg_test = next((r for r in results if r["test"] == "ffmpeg_availability"), None)
ffmpeg_pass = ffmpeg_test and ffmpeg_test["result"] == "PASS"

DISCOVERY_SESSION.update({
    "summary": {
        "total_tests": len(results),
        "passed": passed,
        "failed": failed,
        "unknown": unknown,
        "total_time_s": total_time,
        "critical_ffmpeg_available": ffmpeg_pass,
        "verdict": "PASS" if ffmpeg_pass else "FAIL"
    }
})

output_path = Path("/kaggle/working/discovery_runtime_probe_results.json")
output_path.write_text(json.dumps(DISCOVERY_SESSION, indent=2))

print(f"\n📊 Runtime Packages Summary:")
print(f"   PASS:    {passed}")
print(f"   FAIL:    {failed}")
print(f"   UNKNOWN: {unknown}")
print(f"   Total:   {len(results)} tests in {total_time:.1f}s")
print(f"   FFmpeg (CRITICAL): {'✅ AVAILABLE' if ffmpeg_pass else '❌ MISSING'}")
print(f"\n✅ Results saved to: {output_path}")